# Merge: satellite series + CDL crop type + field metadata

Builds one table for modelling and EDA, keyed on `field_id` × `date` × `sensor`.

| Source | What it adds | Key |
|---|---|---|
| `data/series_barc.parquet` (from `loader.py`) | NDVI, NDTI, NDMI, NDRE and band reflectance per field and date | `field_id`, `date`, `sensor` |
| `data/weather_barc.parquet` (from `loader.py`) | rain in the 3 days before each reading | `field_id`, `date` |
| USDA Cropland Data Layer (CDL) | crop type per field and year, share of corn pixels | `field_id`, `year` |
| BARC GeoPackage | farm, area, centroid | `field_id` |

Grain/Silage labels are **not** in the merged table. They are joined only in the last section, for validation.

The series parquet must exist first. From the repository root:

```
python preprocessing/loader.py --gpkg 04_Evaluation/BARC_Corn.2021-2023.gpkg --years 2021 2022 2023 --months 3 12 --weather
```

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.features import rasterize

# Add the repo root to the import path so the notebook works from any location
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "preprocessing" / "loader.py").exists())
sys.path.insert(0, str(ROOT))
from preprocessing.loader import add_rain_flag, barc_fields, barc_labels, load_series

GPKG = ROOT / "04_Evaluation" / "BARC_Corn.2021-2023.gpkg"
DATA = ROOT / "data"
YEARS = [2021, 2022, 2023]

## 1. Field metadata

In [2]:
fields = barc_fields(GPKG)
utm = fields.to_crs(fields.estimate_utm_crs())
centroid_ll = utm.geometry.centroid.to_crs(4326)

fields_df = (pd.DataFrame(fields.drop(columns="geometry"))[["field_id", "FARM", "FIELD"]]
             .rename(columns={"FARM": "farm", "FIELD": "field_name"})
             .assign(area_ha=utm.geometry.area.values / 1e4,
                     lon=centroid_ll.x.values, lat=centroid_ll.y.values))
print(fields_df.shape)
fields_df.head()

(83, 6)


,field_id,farm,field_name,area_ha,lon,lat
0,1,APU,1,6.637393,-76.893944,39.014721
1,4,APU,1-97,4.209308,-76.895553,39.013455
2,5,APU,1-98,3.444763,-76.891892,39.013352
3,14,APU,18,2.555004,-76.884025,39.018472
4,22,APU,28,6.008031,-76.878718,39.020104


## 2. CDL crop type per field and year

Source: USDA NASS Cropland Data Layer, served by the USDA SCINet image service behind
[CroplandCROS](https://croplandcros.scinet.usda.gov/) (30 m, one map per year).
The window around BARC is downloaded once per year and cached in `data/cdl/`.

For each field and year: the most common class among the pixels whose centre falls inside the polygon,
and the share of corn pixels. CDL is a crop-type map, not a harvest-type label.

In [3]:
CDL_SERVICE = "https://pdi.scinet.usda.gov/image/rest/services/CDL_WM/ImageServer"
CDL_CRS = 5070            # CONUS Albers, the native CDL projection
CDL_RES = 30
# Corn, plus double-crop classes that include corn (small grain followed by corn is common for silage)
CORN_CODES = {1: "corn", 225: "winter wheat/corn", 226: "oats/corn", 237: "barley/corn", 241: "corn/soybeans"}


def cdl_class_names():
    feats = requests.get(f"{CDL_SERVICE}/rasterAttributeTable", params={"f": "json"}, timeout=120).json()["features"]
    return {f["attributes"]["Value"]: f["attributes"]["Class_Names"] for f in feats}


def cdl_raster_ids():
    feats = requests.get(f"{CDL_SERVICE}/query", params={"where": "1=1", "outFields": "OBJECTID,Year,Name",
                         "returnGeometry": "false", "f": "json"}, timeout=120).json()["features"]
    return {f["attributes"]["Year"]: f["attributes"]["OBJECTID"] for f in feats}


def fetch_cdl(year, bounds, raster_ids, cache_dir=DATA / "cdl"):
    # Download (or read from cache) the CDL window for one year. Returns (array, affine transform).
    cache_dir.mkdir(parents=True, exist_ok=True)
    path = cache_dir / f"cdl_{year}.tif"
    if not path.exists():
        x0, y0, x1, y1 = bounds
        params = {"bbox": f"{x0},{y0},{x1},{y1}", "bboxSR": CDL_CRS, "imageSR": CDL_CRS,
                  "size": f"{(x1 - x0) // CDL_RES},{(y1 - y0) // CDL_RES}", "format": "tiff", "pixelType": "U8",
                  "interpolation": "RSP_NearestNeighbor", "noData": 0, "f": "image",
                  "mosaicRule": json.dumps({"mosaicMethod": "esriMosaicLockRaster",
                                            "lockRasterIds": [raster_ids[year]]})}
        r = requests.get(f"{CDL_SERVICE}/exportImage", params=params, timeout=300)
        r.raise_for_status()
        if "tiff" not in r.headers.get("content-type", ""):
            raise RuntimeError(f"CDL {year}: unexpected response {r.text[:200]}")
        path.write_bytes(r.content)
    with rasterio.open(path) as ds:
        return ds.read(1), ds.transform


def cdl_by_field(fields, years):
    proj = fields.to_crs(CDL_CRS)
    b = proj.total_bounds
    pad = 10 * CDL_RES     # snap the window to the 30 m grid, with a margin
    bounds = (int(b[0] // CDL_RES * CDL_RES - pad), int(b[1] // CDL_RES * CDL_RES - pad),
              int(b[2] // CDL_RES * CDL_RES + pad), int(b[3] // CDL_RES * CDL_RES + pad))
    names, ids = cdl_class_names(), cdl_raster_ids()
    rows = []
    for year in years:
        arr, transform = fetch_cdl(year, bounds, ids)
        for fid, geom in zip(proj["field_id"], proj.geometry):
            mask = rasterize([(geom, 1)], out_shape=arr.shape, transform=transform, fill=0, dtype="uint8")
            if not mask.any():      # field smaller than a pixel: take every touched pixel
                mask = rasterize([(geom, 1)], out_shape=arr.shape, transform=transform, fill=0,
                                 dtype="uint8", all_touched=True)
            codes, counts = np.unique(arr[mask == 1], return_counts=True)
            top = int(codes[counts.argmax()])
            n = int(counts.sum())
            rows.append({"field_id": fid, "year": year, "cdl_n_pix": n,
                         "cdl_code": top, "cdl_class": names.get(top, str(top)),
                         "cdl_class_frac": counts.max() / n,
                         "cdl_corn_frac": counts[np.isin(codes, list(CORN_CODES))].sum() / n})
    out = pd.DataFrame(rows)
    out["cdl_is_corn"] = out["cdl_corn_frac"] >= 0.5
    return out


cdl = cdl_by_field(fields, YEARS)
cdl.to_parquet(DATA / "cdl_barc.parquet", index=False)
print(cdl.shape)
cdl.head()

(249, 8)


,field_id,year,cdl_n_pix,cdl_code,cdl_class,cdl_class_frac,cdl_corn_frac,cdl_is_corn
0,1,2021,74,1,Corn,0.621622,0.621622,True
1,4,2021,46,121,Developed/Open Space,0.913043,0.000000,False
2,5,2021,40,5,Soybeans,0.675000,0.050000,False
3,14,2021,30,176,Grassland/Pasture,0.766667,0.000000,False
4,22,2021,67,1,Corn,0.716418,0.716418,True


In [4]:
# Most common CDL class per year (number of fields)
cdl.groupby(["year", "cdl_class"]).size().unstack(0).fillna(0).astype(int).sort_values(2023, ascending=False)

year,2021,2022,2023
cdl_class,,,
Soybeans,28,33,31
Corn,30,18,29
Grassland/Pasture,11,15,14
Winter Wheat,2,1,3
Dbl Crop WinWht/Soybeans,6,11,2
Other Hay/Non Alfalfa,3,1,2
Developed/Open Space,1,1,1
Developed/Med Intensity,1,1,1
Barley,0,1,0


## 3. Satellite series (NDVI and other measures)

`load_series` returns usable readings only (at least 60% of the field clear).
Sentinel-2 and Landsat NDVI differ by about 0.05, so keep `sensor` when comparing values.

In [5]:
series_path = DATA / "series_barc.parquet"
if not series_path.exists():
    raise FileNotFoundError(f"{series_path} not found. Run preprocessing/loader.py first (see the top of this notebook).")

series = load_series(series_path)
weather_path = DATA / "weather_barc.parquet"
if weather_path.exists():
    series = add_rain_flag(series, pd.read_parquet(weather_path))

print(series.shape, "| fields:", series["field_id"].nunique(),
      "| dates:", series["date"].min().date(), "to", series["date"].max().date())
series.groupby(["year", "sensor"]).size().unstack()

(14689, 26) | fields: 83 | dates: 2021-03-04 to 2023-12-31


sensor,LANDSAT,S2
year,,
2021,1180,3684
2022,1436,3426
2023,1619,3344


## 4. Merge

One row per field, date and sensor. Field metadata joins on `field_id`; CDL joins on `field_id` and `year`.

In [6]:
merged = (series
          .merge(fields_df, on="field_id", how="left", validate="many_to_one")
          .merge(cdl, on=["field_id", "year"], how="left", validate="many_to_one"))
merged["doy"] = merged["date"].dt.dayofyear

assert len(merged) == len(series)
assert merged["cdl_code"].notna().all() and merged["farm"].notna().all()

merged.to_parquet(DATA / "merged_barc.parquet", index=False)
print(merged.shape)
merged[["field_id", "year", "date", "doy", "sensor", "ndvi", "ndti", "ndmi", "ndre",
        "farm", "area_ha", "cdl_class", "cdl_corn_frac", "cdl_is_corn"]].head()

(14689, 38)


,field_id,year,date,doy,sensor,ndvi,ndti,ndmi,ndre,farm,area_ha,cdl_class,cdl_corn_frac,cdl_is_corn
0,1,2021,2021-03-10,69,LANDSAT,0.365914,0.165990,-0.077693,NaN,APU,6.637393,Corn,0.621622,True
1,1,2021,2021-03-11,70,S2,0.334014,0.146249,-0.084456,0.208350,APU,6.637393,Corn,0.621622,True
2,1,2021,2021-03-14,73,S2,0.352973,0.141320,-0.099569,0.222511,APU,6.637393,Corn,0.621622,True
3,1,2021,2021-03-19,78,S2,0.383390,0.181883,-0.019200,0.247747,APU,6.637393,Corn,0.621622,True
4,1,2021,2021-03-21,80,S2,0.381603,0.163221,-0.051975,0.244386,APU,6.637393,Corn,0.621622,True


In [7]:
# Coverage: usable readings per field and year
per_field = merged.groupby(["year", "field_id"]).size()
per_field.groupby("year").describe()[["count", "min", "50%", "max"]]

,count,min,50%,max
year,,,,
2021,83.0,37.0,62.0,81.0
2022,83.0,42.0,52.0,78.0
2023,83.0,43.0,53.0,84.0


## 5. Labels (validation and EDA only)

Labels are joined here, on `field_id` and `year`, and kept out of `merged_barc.parquet`.
Blank labels are not negative examples, so only labelled field-years are kept.

In [8]:
labels = barc_labels(GPKG)
labeled = merged.merge(labels, on=["field_id", "year"], how="inner", validate="many_to_one")
print(labeled.shape, "| labelled field-years with readings:",
      labeled[["field_id", "year"]].drop_duplicates().shape[0], "of", len(labels))

# Does CDL agree that the labelled field-years are corn?
check = labels.merge(cdl, on=["field_id", "year"], how="left")
pd.crosstab([check["year"], check["harvest_type"]], check["cdl_class"])

(2750, 39) | labelled field-years with readings: 44 of 44


cdl_class          Corn  Soybeans
year harvest_type                
2021 Grain           14         0
     Silage           4         0
2022 Grain            1         2
     Silage           4         0
2023 Grain           11         4
     Silage           4         0

In [9]:
# Median NDVI by month and harvest type, Sentinel-2 only
s2 = labeled[labeled["sensor"] == "S2"]
(s2.assign(month=s2["date"].dt.month)
   .groupby(["month", "harvest_type"])["ndvi"].median().unstack().round(2))

harvest_type,Grain,Silage
month,,
3,0.38,0.59
4,0.50,0.71
5,0.23,0.24
6,0.46,0.53
7,0.82,0.87
8,0.83,0.86
9,0.57,0.28
10,0.36,0.30
11,0.36,0.67
